In [ ]:
#  START
#    ↓
# Product
#    ↓
# Product Intelligence
#    ↓
# ProductProfile
#    ↓
#  HITL   
#    ↓
# Product Intelligence
#   FAN OUT / CHOOSE ONE AGENT 
#    ├───────────────┐
#    ↓               ↓
# SBOM            Vendor Risk
#    ↓               ↓
# CVE Analysis   Vendor Analysis
#    └───────┬───────┘
#            ↓
#       Risk Aggregator
#            ↓
#       ProductRisk
#           ↓
#          HITL 


In [ ]:
import os 
import requests 
import json 

from pydantic import BaseModel , Field 
from typing import List, Dict, Any, Optional, Annotated, Literal
from langchain.agents import create_agent 
from langchain_openai import ChatOpenAI
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, Interrupt
from langchain_tavily import TavilySearch
from langchain_tavily import TavilyCrawl
from langchain.messages import AIMessage, AIMessageChunk, AnyMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

True

In [ ]:
def chat_openai_client(reasoning : Literal['low', 'medium', 'high']):
    return ChatOpenAI(
        api_key = os.environ['OPENAI_API_KEY'],
        model = "gpt-5-nano",
        reasoning_effort = reasoning,
        max_retries = 5
    )


def decision_client(payload : Dict):
    url="https://openrouter.ai/api/alpha/decisions",
    headers={
        "Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
        "Content-Type": "application/json",
    }
    model = "inception/mercury-decide:free"
    payload['model'] = model 

    response = requests.post(
        url = url,
        headers= headers,
        data = payload
    )

    if response.status_code == 404:
        return {
            "status" : "failed",
            "message" : response.reason,
            "response" : None 
        }
    return {
        "status" : "success",
        "response" : response
    }

def tavily_client(type : Literal['search', 'crawl']) -> TavilyCrawl | TavilySearch:
    api_key = os.environ['TAVILY_API_KEY']

    if type == 'search':
        return TavilySearch(
            max_results=5,
            topic="general",
            api_key = api_key
        )

    else:
        raise NotImplementedError


In [ ]:
class ProductProfile(BaseModel):
    product_name : str
    vendor : str
    version : str
    category : Literal["Developer_Tool", "Open_Source"] 
    repository : Optional[str] = None 
    deployment : Literal["Cloud", "On-prem"]


class VendorAnalysisReport(BaseModel):
    pass 

class CVE_Report(BaseModel):
    pass 

class ComplianceReport(BaseModel):
    pass 

class ThreatIntelligenceReport(BaseModel):
    pass 

class ProductRiskAssessmentState(BaseModel):
    query : str 
    product_profile : Optional[ProductProfile | None] = None 

    vendor_report : Optional[VendorAnalysisReport | None] = None 
    cve_report : Optional[CVE_Report | None] = None 
    compliance_report : Optional[ComplianceReport | None] = None 
    threat_intelligence_report : Optional[ThreatIntelligenceReport | None] = None 
    
    
    # Run Time Vars 
    messages : List[Annotated, Dict[str, Any]] = []
    status : Literal["running", "paused", "failed", "completed"]


In [ ]:
def crawl_profile_details(user_query : str):
  tool = tavily_client(type = "search")
  product_details = tool.invoke({"query": user_query})
  return json.dumps(product_details)


def product_profile_gen_node(state : ProductRiskAssessmentState, config : Dict):
    if not state.user_query:
       return {
          "status" : "failed"
       }
    
    reasoning_payload = json.dumps({
        "state" : state.user_query,
        "questions" : {
            "reasoning" : {
              "type" : "choice",
              "instructions" : "Which reasoning efforts must be selected for this user query ?"
              "criteria": {
                "low": " ",
                "medium": " ",
                "high" : " "
              }
            }}
        })
    reasoning_decision = decision_client(reasoning_payload)
    reasoning_answer = reasoning_decision.json()["answers"]
    probs = max(reasoning_answer["reasoning"]["probabilities"])
    max_class = reasoning_answer["reasoning"]["choice"] if probs in reasoning_answer["reasoning"]["probabilities"]

    llm = chat_openai_client(reasoning= max_class)
    structured_llm = llm.with_structured_output(ProductProfile)

    payload = json.dumps({
        "state" : state.user_query,
        "questions" : {
            "is_urgent" : {
              "type" : "noul",
              "instructions" : "Does this user query needs more profile details. "
              "Do we call crawl_profile_details tool_call ?",
              "criteria": {
                "true": "Because the current details are insufficient",
                "false": "The User mentioned details are sufficient"
              }}}
        })
    decision = decision_client(payload)
    answer = decision.json()["answers"]

    if answer["is_urgent"]["noul"] ==  1:
        product_profile = crawl_profile_details(state.user_query)      

    report = structured_llm.invoke(
    f"""
    Analyze these search results and produce a Product Profile.
    ## User Query:
    {state.user_query}

    ## Search results:
    {product_profile}
    """
)    
      
    return {
       "product_profile" : report,
       "status": "running"
    }



In [ ]:
# Interrupt Node
def standalone_interrupt_node(state : ProductRiskAssessmentState, next_route_to : str, config: Dict):

    decision = interrupt({
        "type" : "analysis_approval",
        "description" : "Approve | Reject the "
    })

    return {
        "route_to" : " ",
        "messages" : []
    }

In [ ]:
# Decision Agent - Classifier (Compulsary after Aggregation Node)
def decision_agent_node(state : ProductRiskAssessmentState, next_route_to : str, config : Dict):

    return {
        "route_to" : ""
    }

In [ ]:
# Stadalone Vendor Risk Analyzer 
def vendor_risk_analysis_node(state : ProductRiskAssessmentState, next_route_to : str, config: Dict):
    
    return {
        "route_to" : "",
        "messages" : [{"role" : "assistant", "message" : ""}]
    } 

In [ ]:
# Standalone Common Vulnerability Analyzer
def cve_analysis_node(state: ProductRiskAssessmentState, config : Dict):

    return {
        "messages" : [] 
    }  

In [ ]:
def compliance_validator_node(state : ProductRiskAssessmentState, next_route_to : str, config : Dict):
    pass 

In [ ]:
# Risk Aggregator 
def risk_aggregator_node(state : ProductRiskAssessmentState, next_route_to : str, config : Dict):
    return {
        "route_to" : "" 
    }

In [ ]:
 # SAMPLE INPUT 
""" 
blueprint = [
    {
        "node": {
            "node_name": "product_profile",
            "node_route": "node_001"
        }
    },
    {
        "node": {
            "node_name": "cve_analysis",
            "node_route": "node_002"
        }
    },
    {
        "node": {
            "node_name": "vendor_analysis",
            "node_route": "node_003"
        }
    },
    {
        "node": {
            "node_name": "product_risk",
            "node_route": "node_004"
        }
    },
    {
        "edge": {
            "edge_type": "fan_out",
            "source": "node_001",
            "targets": [
                "node_002",
                "node_003"
            ]
        }
    },
    {
        "edge": {
            "edge_type": "fan_in",
            "sources": [
                "node_002",
                "node_003"
            ],
            "target": "node_004"
        }
    }
]
"""

In [ ]:
PLUGIN_REGISTRY = {
    "product_profile": product_profile_agent,
    "cve_analysis": cve_analysis_agent,
    "vendor_analysis": vendor_analysis_agent,
    "compliance": compliance_agent,
    "threat_intelligence": threat_intelligence_agent,
    "product_risk": product_risk_agent,
}

def function_mapper(node_name: str):
    try:
        return PLUGIN_REGISTRY[node_name]
    except KeyError:
        raise ValueError(f"Unknown plugin: {node_name}")

In [4]:
from typing import Any


def parse_blueprint(
    blueprint: list[dict[str, Any]]
) -> dict[str, list[dict[str, str]]]:

    nodes: list[dict[str, str]] = []
    edges: list[dict[str, str]] = []

    previous_node: str | None = None

    fan_out_source: str | None = None
    fan_out_nodes: list[str] = []

    waiting_for_fan_in_target = False
    fan_in_sources: list[str] = []

    for event in blueprint:

        # ==================================================
        # NODE
        # ==================================================

        if "node" in event:

            payload = event["node"]

            node_name = payload["node_name"]
            node_route = payload["node_route"]

            nodes.append({
                "name": node_name,
                "route": node_route
            })

            # ----------------------------------------------
            # First node after FAN-IN
            # ----------------------------------------------

            if waiting_for_fan_in_target:

                for source in fan_in_sources:

                    edges.append({
                        "source": source,
                        "target": node_route
                    })

                waiting_for_fan_in_target = False
                fan_in_sources = []

                previous_node = node_route

            # ----------------------------------------------
            # Node inside FAN-OUT
            # ----------------------------------------------

            elif fan_out_source is not None:

                fan_out_nodes.append(node_route)

            # ----------------------------------------------
            # Normal node → node
            # ----------------------------------------------

            elif previous_node is not None:

                edges.append({
                    "source": previous_node,
                    "target": node_route
                })

                previous_node = node_route

            # ----------------------------------------------
            # First node
            # ----------------------------------------------

            else:

                previous_node = node_route

        # ==================================================
        # FAN OUT
        # ==================================================

        elif "fan_out" in event:

            if previous_node is None:
                raise ValueError(
                    "fan_out cannot appear before a node"
                )

            fan_out_source = previous_node
            fan_out_nodes = []

        # ==================================================
        # FAN IN
        # ==================================================

        elif "fan_in" in event:

            if fan_out_source is None:
                raise ValueError(
                    "fan_in found without active fan_out"
                )

            if not fan_out_nodes:
                raise ValueError(
                    "fan_in found but fan_out has no branches"
                )

            # Save branches as sources of fan-in
            fan_in_sources = fan_out_nodes.copy()

            # Connect source → branches
            for target in fan_out_nodes:

                edges.append({
                    "source": fan_out_source,
                    "target": target
                })

            fan_out_source = None
            fan_out_nodes = []

            waiting_for_fan_in_target = True

        # ==================================================
        # UNKNOWN EVENT
        # ==================================================

        else:

            raise ValueError(
                f"Unknown blueprint event: {event}"
            )

    # ------------------------------------------------------
    # Validation
    # ------------------------------------------------------

    if fan_out_source is not None:
        raise ValueError(
            "Blueprint ended before fan_out was closed"
        )

    if waiting_for_fan_in_target:
        raise ValueError(
            "Blueprint ended after fan_in without target node"
        )

    return {
        "nodes": nodes,
        "edges": edges
    }

In [6]:
# product_profile → cve_analysis
# product_profile → vendor_analysis

# cve_analysis    → product_risk
# vendor_analysis → product_risk

# {
#     "nodes": [
#         {
#             "name": "product_profile",
#             "route": "product_profile"
#         },
#         {
#             "name": "cve_analysis",
#             "route": "cve_analysis"
#         },
#         {
#             "name": "vendor_analysis",
#             "route": "vendor_analysis"
#         },
#         {
#             "name": "product_risk",
#             "route": "product_risk"
#         }
#     ],

#     "edges": [
#         {
#             "source": "product_profile",
#             "target": "cve_analysis"
#         },
#         {
#             "source": "product_profile",
#             "target": "vendor_analysis"
#         },
#         {
#             "source": "cve_analysis",
#             "target": "product_risk"
#         },
#         {
#             "source": "vendor_analysis",
#             "target": "product_risk"
#         }
#     ]
# }


In [8]:
def find_start_nodes(parsed_blueprint : Dict[str, List[Dict[str, Any]]]):
    edges = parsed_blueprint.get("edges")
    if not edges:
        return "Error"
    start_ = edges[0] 
    return start_.get("source")

def find_end_nodes(parsed_blueprint : Dict[str, List[Dict[str, Any]]]):
    edges = parsed_blueprint.get("edges")
    if not edges:
        return "Error"
    end_ = edges[-1]
    return end_.get("target")

In [ ]:
def graph_builder(
    blueprint: list[dict[str, Any]]
) -> StateGraph:

    parsed = parse_blueprint(blueprint)
    checkpointer = InMemorySaver()

    builder = StateGraph(
        ProductRiskAssessmentState
    )

    # Add nodes
    for node in parsed["nodes"]:

        plugin_func = function_mapper(
            node["name"]
        )

        builder.add_node(
            node["route"],
            plugin_func
        )

    # Add edges
    for edge in parsed["edges"]:

        builder.add_edge(
            edge["source"],
            edge["target"]
        )

    # START
    start_nodes = find_start_nodes(
        parsed["nodes"],
        parsed["edges"]
    )

    for node in start_nodes:
        builder.add_edge(
            START,
            node
        )

    # END
    end_nodes = find_end_nodes(
        parsed["nodes"],
        parsed["edges"]
    )

    for node in end_nodes:
        builder.add_edge(
            node,
            END
        )

    return builder.compile(checkpointer = checkpointer)

In [ ]:
def render_interrupt(interrupt : Interrupt):
    interrupts= interrupt.value
    for action in interrupts["action_requests"]:
        print(action['description'])

def render_message(token : AIMessageChunk):
    if token.text:
        print(token.text, end = '|')
    if token.tool_call_chunks:
        print(token.tool_call_chunks)

In [ ]:
import uuid 
agent = None 
thread_id = uuid.uuid8()

user_query = """ 

"""

config = {"configurable" : {"thread_id": thread_id}}
interrupts = []

for event in agent.astream(
    {"messages" : [{"role" : "user", "content" : user_query}]},
    streaming_mode = ["messages", "updates"],
    version = "v2",
):
    if event["type"] == "messages":
        token, metadata = event["data"]
        if isinstance(token, AIMessageChunk):
            render_message(token = token)
        

    if event["type"] == "updates":
        for source, update in event["data"].items():
            if source == "__interrupt__":
                interrupts.extend(update)
                render_interrupt(update[0])